# 01 — Loop 1 Walking Skeleton

End-to-end pipeline: **load → split → preprocess → train → AUC-PR**.

This is the **v0.1 Loop 1 integrator** — the last win condition for `rp-prey-001`.
Its job is to prove the full pipeline composes cleanly, not to produce tuned metrics.

**Data source priority:**
1. BigQuery `pa-warehouse-prod.marts.v_attrition_features` (live; requires SA key).
2. Snapshot CSV at `data/raw/v_attrition_features.csv` (if present from a prior BQ load).
3. Synthetic in-memory dataset (guaranteed to work in CI / fresh-clone reviews).

**Win condition:** notebook runs green via `pytest --nbmake` and produces an
AUC-PR score with a Rung 1 caption.

**Epistemic note (Rung 1):** This model measures statistical association between
HRIS signals and voluntary exits. It does **not** identify causal drivers —
that requires the causal slice (Epic 7, v1.1 roadmap). Treat the output as a
ranked risk list, not a causal claim.

In [ ]:
# Cell 1 — Scaffolding
from retention import config

config.set_global_seed()
config.configure_plot_style()

print(f"SEED = {config.SEED}")
print(f"PROJECT_ROOT = {config.PROJECT_ROOT}")

In [ ]:
# Cell 2 — Load data
#
# Priority: BQ live → CSV snapshot → synthetic fallback.
# The synthetic fallback is intentionally simple — just enough signal for XGBoost
# to learn something above random, so AUC-PR > naive baseline is verifiable.
import os
from pathlib import Path

import numpy as np
import pandas as pd

_SA_KEY = Path(os.getenv("PA_WAREHOUSE_SA_KEY", str(Path.home() / ".gcp" / "pa-warehouse-sa.json")))
_CSV_PATH = config.DATA_DIR / "raw" / "v_attrition_features.csv"

df: pd.DataFrame | None = None
_data_source = ""

# --- Attempt 1: BigQuery ---
if _SA_KEY.exists():
    try:
        from retention.data.load import load_attrition_features
        df = load_attrition_features(
            project_id=config.BQ_PROJECT_ID,
            dataset=config.BQ_DATASET_MARTS,
            table=config.BQ_TABLE_ATTRITION_FEATURES,
            snapshot_to_csv=True,
            snapshot_dir=config.DATA_DIR / "raw",
            max_bytes_billed=100 * 1024 * 1024,
            dry_run=False,
            sa_key=_SA_KEY,
            enforce_contract=True,
        )
        _data_source = "BigQuery live"
    except Exception as exc:
        print(f"BQ load failed ({type(exc).__name__}: {exc}); trying CSV fallback.")

# --- Attempt 2: CSV snapshot ---
if df is None and _CSV_PATH.exists():
    from retention.data.load import load_attrition_features_local
    df = load_attrition_features_local(_CSV_PATH, enforce_contract=True)
    _data_source = f"CSV snapshot ({_CSV_PATH.name})"

# --- Attempt 3: Synthetic fallback ---
if df is None:
    print("No BQ credentials and no CSV snapshot found.")
    print("Using synthetic in-memory dataset (CI / fresh-clone path).")
    _rng = np.random.default_rng(config.SEED)
    N = 1_000
    _tenure = _rng.uniform(1, 72, N)
    _compa = _rng.uniform(0.7, 1.4, N)
    _perf = _rng.choice(["2", "3", "4", "5"], N)
    _gender = _rng.choice(["M", "F", "NB"], N)
    _is_crit = _rng.choice([True, False], N)
    _succ = _rng.integers(0, 5, N).astype(float)
    _age = _rng.uniform(22, 62, N)
    # Baked signal: low compa + low tenure → higher exit prob
    _prob = 0.05 + 0.3 * (_compa < 0.85) + 0.15 * (_tenure < 12)
    _label = _rng.uniform(size=N) < _prob
    df = pd.DataFrame({
        "employee_id": [f"EMP_{i:05d}" for i in range(N)],
        "snapshot_date": pd.to_datetime(["2025-05-27"] * N),
        "tenure_months": _tenure,
        "performance_tier": _perf,
        "compa_ratio": _compa,
        "is_critical_role": _is_crit,
        "successor_count": _succ,
        "age_at_window_close": _age,
        "gender": _gender,
        "voluntary_exit_label": _label,
    })
    _data_source = "synthetic in-memory (CI fallback)"

print(f"\nData source : {_data_source}")
print(f"Shape       : {df.shape}")
print(f"Exit rate   : {df['voluntary_exit_label'].mean():.2%}")
df.head(3)

In [ ]:
# Cell 3 — Temporal split
#
# Splits chronologically: oldest 70% → train, next 15% → val, newest 15% → test.
# When all rows share a single snapshot_date (as in the v0.1 mart), the sort is
# deterministic via the employee_id tiebreak — the no-leak invariant holds vacuously.
# Loop 2 will address the single-snapshot limitation (see BACKLOG.md Preconditions).
from retention.data.split import temporal_split

train, val, test = temporal_split(df, save_indices=False)

print(f"train : {len(train):,} rows ({len(train)/len(df):.1%})  exit rate {train['voluntary_exit_label'].mean():.2%}")
print(f"val   : {len(val):,} rows ({len(val)/len(df):.1%})  exit rate {val['voluntary_exit_label'].mean():.2%}")
print(f"test  : {len(test):,} rows ({len(test)/len(df):.1%})  exit rate {test['voluntary_exit_label'].mean():.2%}")

In [ ]:
# Cell 4 — Prepare X / y splits
#
# Drop identifier + temporal anchor + label columns from feature matrix.
# The ColumnTransformer uses remainder='drop' anyway, but being explicit
# avoids shape surprises and makes the data contract visible.
from retention.features.catalog import (
    get_identifier_names,
    get_label_name,
    get_temporal_anchor,
)
from retention.features.preprocessing import get_feature_columns

_drop_cols = get_identifier_names() + [get_temporal_anchor()]
_label_col = get_label_name()
_feature_cols = get_feature_columns()

X_train = train[_feature_cols]
y_train = train[_label_col].astype(int)

X_val = val[_feature_cols]
y_val = val[_label_col].astype(int)

X_test = test[_feature_cols]
y_test = test[_label_col].astype(int)

print(f"Feature columns ({len(_feature_cols)}): {_feature_cols}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train positives: {y_train.sum()} ({y_train.mean():.2%})")

In [ ]:
# Cell 5 — Preprocess (fit on train only)
#
# The ColumnTransformer is built from FEATURE_CATALOG at import time;
# fit_transform on X_train only — no val/test data ever touches the imputer/scaler fit.
from retention.features.preprocessing import build_preprocessor, get_feature_names_out

preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

_out_names = get_feature_names_out(preprocessor)
print(f"Output features ({len(_out_names)}): {_out_names}")
print(f"X_train_t shape: {X_train_t.shape}")
print(f"NaN in X_train_t: {np.isnan(X_train_t).any()}")
print(f"NaN in X_test_t : {np.isnan(X_test_t).any()}")

In [ ]:
# Cell 6 — Train XGBoost (val set monitoring)
#
# RetentionModel handles the fit internally: preprocessor fit on train,
# eval_set computed from val, XGBClassifier tracks aucpr per round.
# n_estimators=100 is a v0.1 default — Loop 2 will tune via optuna.
from retention.models.xgb import RetentionModel

model = RetentionModel(n_estimators=100)
model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

print("RetentionModel fitted.")
print(f"Pipeline steps: {[name for name, _ in model.pipeline.steps]}")

In [ ]:
# Cell 7 — Evaluate on test set
#
# AUC-PR is the primary metric for this project (imbalanced classes, business
# cares about ranking exits, not overall accuracy).
# The Rung 1 caption is a permanent epistemic flag — association, not causation.
from retention.evaluation.metrics import auc_pr, format_rung1_caption

y_proba_test = model.predict_proba(X_test)[:, 1]
score = auc_pr(y_test, y_proba_test)
caption = format_rung1_caption(score)

# Naive baseline: a constant predictor scores exactly class prevalence
_baseline = float(y_test.mean())
_lift = score / _baseline if _baseline > 0 else float("inf")

print(f"Test set results")
print(f"  {caption}")
print(f"  Naive baseline (prevalence)  = {_baseline:.3f}")
print(f"  Lift over naive baseline     = {_lift:.2f}×")

# Sanity check: a model with learned signal should beat the baseline
# (on synthetic data with baked correlations this should hold reliably)
assert score > _baseline * 0.9, (
    f"AUC-PR ({score:.3f}) is substantially below naive baseline ({_baseline:.3f}). "
    f"Model may be degenerate — check data or features."
)

In [ ]:
# Cell 8 — Precision-Recall curve
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(
    y_test,
    y_proba_test,
    name=f"XGBoost v0.1",
    ax=ax,
)
ax.axhline(y=_baseline, color="grey", linestyle="--", linewidth=1, label=f"Naive baseline ({_baseline:.3f})")
ax.set_title(f"Precision-Recall Curve\n{caption}")
ax.legend()
plt.tight_layout()

_report_path = config.REPORTS_DIR / "figures" / "loop1_pr_curve.png"
_report_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(_report_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_report_path}")

In [ ]:
# Cell 9 — Top-risk employees (actionable output)
#
# What HR actually receives: a ranked list of employees with exit probability.
# This demonstrates the model produces a deployable artifact, not just a metric.
y_proba_full = model.predict_proba(df[_feature_cols])[:, 1]
risk_df = df[["employee_id"]].copy()
risk_df["exit_probability"] = y_proba_full
risk_df["actual_exit"] = df[_label_col].astype(int).values
risk_df = risk_df.sort_values("exit_probability", ascending=False).reset_index(drop=True)

print("Top 10 at-risk employees (model output):")
print(risk_df.head(10).to_string(index=True))
print(f"\nPrecision @ top-10: {risk_df.head(10)['actual_exit'].mean():.0%} of flagged employees actually exited")

## Loop 1 Summary

| Step | Status |
|------|--------|
| Data load (BQ / CSV / synthetic) | ✅ |
| Temporal split (70/15/15) | ✅ |
| Preprocessing (ColumnTransformer) | ✅ |
| XGBoost training (eval_set monitoring) | ✅ |
| AUC-PR evaluation + Rung 1 caption | ✅ |
| PR curve saved to reports/figures/ | ✅ |
| Top-risk ranked list | ✅ |

**v0.1 ships here.** This notebook is the walking skeleton —
every future loop adds depth to one or more of these steps.

**Loop 2 next steps** (rp-prey-002):
- Dual cohort (HRIS-only vs hybrid) when `paw-prey-NNN` delivers survey columns
- Hyperparameter tuning via Optuna
- Nested cross-validation
- Temporal anchor fix if pa-warehouse delivers multi-snapshot data

---
*Pipeline end — `pytest --nbmake notebooks/01_loop1_walking_skeleton.ipynb` should pass.*